In [ ]:
import os
print(os.getcwd())

In [ ]:
!ls

In [ ]:
!ls

In [ ]:
¡Perfecto! Con los cimientos de LocalStack estables y la comunicación al 100%, el siguiente paso es conectar tu lógica de aplicación a estos endpoints simulados.

Para lograr una integración limpia sin alterar el código que eventualmente irá a producción (AWS real), el enfoque ideal es usar una arquitectura basada en variables de entorno. De esta manera, mantienes la flexibilidad de nuestro marco de equilibrio: el mismo código funciona de manera idéntica a nivel local y en la nube, solo balanceando sus puntos de conexión.

Aquí tienes la estrategia secuencial para proceder con el despliegue e integración:

1. Configurar las Variables de Entorno
La aplicación necesita saber cuándo apuntar a AWS real y cuándo desviar el tráfico hacia el puerto 4566 de LocalStack. Define estas variables en tu entorno de desarrollo o en un archivo .env:

Bash


# Configuración para el entorno local (Fase 5)
APP_ENV=local
AWS_ENDPOINT_URL=http://localhost:4566
AWS_DEFAULT_REGION=us-east-1
AWS_ACCESS_KEY_ID=test
AWS_SECRET_ACCESS_KEY=test
S3_BUCKET_NAME=fase5-storage-bucket
DYNAMODB_TABLE_NAME=fase5-execution-locks

In [ ]:
# Configuración para el entorno local (Fase 5)
APP_ENV=local
AWS_ENDPOINT_URL=http://localhost:4566
AWS_DEFAULT_REGION=us-east-1
AWS_ACCESS_KEY_ID=test
AWS_SECRET_ACCESS_KEY=test
S3_BUCKET_NAME=fase5-storage-bucket
DYNAMODB_TABLE_NAME=fase5-execution-locks

In [ ]:
import os
import boto3

# Leer la configuración del entorno
env = os.getenv("APP_ENV", "local")
endpoint_url = os.getenv("AWS_ENDPOINT_URL", None) if env == "local" else None

# Inicializar recursos de forma dinámica
def get_s3_client():
    return boto3.client(
        's3',
        endpoint_url=endpoint_url,
        region_name=os.getenv("AWS_DEFAULT_REGION", "us-east-1")
    )

def get_dynamodb_resource():
    return boto3.resource(
        'dynamodb',
        endpoint_url=endpoint_url,
        region_name=os.getenv("AWS_DEFAULT_REGION", "us-east-1")
    )

# Ejemplo de uso en la lógica de la aplicación
s3 = get_s3_client()
dynamodb = get_dynamodb_resource()

In [ ]:
import os
import sys
import boto3
from botocore.exceptions import ClientError

# ==========================================
# CONFIGURACIÓN DE ENTORNO (Eje de Control)
# ==========================================
# Si no existen en el sistema, usamos por defecto tus parámetros de LocalStack
APP_ENV = os.getenv("APP_ENV", "local")
AWS_ENDPOINT_URL = os.getenv("AWS_ENDPOINT_URL", "http://localhost:4566")
AWS_REGION = os.getenv("AWS_DEFAULT_REGION", "us-east-1")

BUCKET_NAME = os.getenv("S3_BUCKET_NAME", "fase5-storage-bucket")
TABLE_NAME = os.getenv("DYNAMODB_TABLE_NAME", "fase5-execution-locks")

# Forzar credenciales dummy si estamos en entorno local para evitar que boto3 busque reales
if APP_ENV == "local":
    os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID", "test")
    os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY", "test")

# Determinar el endpoint dinámicamente según el entorno
ENDPOINT_URL = AWS_ENDPOINT_URL if APP_ENV == "local" else None

print(f"--- Iniciando Integración de Aplicación [Entorno: {APP_ENV.upper()}] ---")
if ENDPOINT_URL:
    print(f"Apuntando a simulador LocalStack en: {ENDPOINT_URL}\n")

# ==========================================
# INICIALIZACIÓN DE CLIENTES DE AWS
# ==========================================
try:
    s3_client = boto3.client('s3', endpoint_url=ENDPOINT_URL, region_name=AWS_REGION)
    dynamodb_resource = boto3.resource('dynamodb', endpoint_url=ENDPOINT_URL, region_name=AWS_REGION)
    dynamo_table = dynamodb_resource.Table(TABLE_NAME)
except Exception as e:
    print(f"[-] Error crítico al inicializar clientes de boto3: {e}")
    sys.exit(1)

# ==========================================
# FLUJO DE VALIDACIÓN SECUENCIAL
# ==========================================
def run_validation_flow():
    test_key = "test_execution_log.json"
    test_content = '{"status": "running", "phase": 5, "msg": "App conectada a localstack_simulador"}'
    
    # Asumimos que la partición (Hash Key) de tu Terraform es LockID
    # Ajusta 'LockID' si tu tabla utiliza otro nombre de atributo clave
    lock_id_attr = "LockID" 
    test_lock_value = "fase5-app-execution-test"

    try:
        # --- PASO 1: Prueba de Escritura en S3 ---
        print("[Paso 1/4] Subiendo archivo de prueba a S3...")
        s3_client.put_object(
            Bucket=BUCKET_NAME,
            Key=test_key,
            Body=test_content,
            ContentType="application/json"
        )
        print(f"  [✓] Archivo '{test_key}' guardado con éxito en '{BUCKET_NAME}'.")

        # --- PASO 2: Lectura y Verificación en S3 ---
        print("[Paso 2/4] Leyendo y verificando archivo desde S3...")
        response = s3_client.get_object(Bucket=BUCKET_NAME, Key=test_key)
        retrieved_content = response['Body'].read().decode('utf-8')
        
        if retrieved_content == test_content:
            print("  [✓] Verificación de integridad exitosa. Los datos coinciden perfectamente.")
        else:
            print("  [X] Advertencia: El contenido recuperado no coincide con el original.")

        # --- PASO 3: Registro de Estado en DynamoDB ---
        print(f"[Paso 3/4] Insertando lock de ejecución en DynamoDB ('{TABLE_NAME}')...")
        dynamo_table.put_item(
            Item={
                lock_id_attr: test_lock_value,
                "Status": "ACTIVE",
                "Timestamp": "2026-07-02T17:30:00Z",
                "Owner": "IntegrationTestScript"
            }
        )
        print(f"  [✓] Registro creado con éxito bajo {lock_id_attr}='{test_lock_value}'.")

        # --- PASO 4: Limpieza Automatizada ---
        print("[Paso 4/4] Ejecutando limpieza del entorno local...")
        
        # Eliminar de S3
        s3_client.delete_object(Bucket=BUCKET_NAME, Key=test_key)
        print(f"  [✓] Objeto '{test_key}' eliminado de S3.")
        
        # Eliminar de DynamoDB
        dynamo_table.delete_item(Key={lock_id_attr: test_lock_value})
        print(f"  [✓] Registro '{test_lock_value}' eliminado de DynamoDB.")

        print("\n=======================================================")
        print("[ÉXITO] Integración completada al 100%. Código balanceado.")
        print("=======================================================")

    except ClientError as ce:
        print(f"\n[!] Error de comunicación con AWS/LocalStack: {ce.response['Error']['Message']}")
    except Exception as e:
        print(f"\n[!] Ocurrió un error inesperado durante las pruebas: {e}")

if __name__ == "__main__":
    run_validation_flow()

In [ ]:
# Dockerfile
FROM python:3.11-slim

WORKDIR /app

# Copiamos los requerimientos e instalamos dependencias
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copiamos el script de integración
COPY app.py .

# Ejecutamos el script por defecto
CMD ["python", "app.py"]

In [ ]:
%%writefile Dockerfile
FROM python:3.11-slim

WORKDIR /app

# Copiamos los requerimientos e instalamos dependencias
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copiamos el script de integración
COPY app.py .

# Ejecutamos el script por defecto
CMD ["python", "app.py"]

In [ ]:
version: '3.8'

services:
  localstack:
    image: localstack/localstack:2.3.2
    container_name: localstack_simulador
    ports:
      - "127.0.0.1:4566:4566"
    environment:
      # Agregamos dynamodb para que el script pueda crear los locks de ejecución
      - SERVICES=s3,sqs,dynamodb  
      - AWS_DEFAULT_REGION=us-east-1
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:4566/_localstack/health"]
      interval: 3s
      timeout: 3s
      retries: 5

  fleet-worker:
    build: .
    image: fleet-multicloud-worker:local
    container_name: fleet_batch_worker_instance
    environment:
      - SIMULACION=False
      - AWS_INPUT_BUCKET=dev-fleet-raw-data
      - AWS_INPUT_KEY=daily_telemetry.csv
      - AZURE_OUTPUT_CONTAINER=dev-fleet-alerts
      - AWS_ACCESS_KEY_ID=mock_key_para_pruebas_locales
      - AWS_SECRET_ACCESS_KEY=mock_secret_para_pruebas_locales
      - AWS_ENDPOINT_URL=http://localstack:4566
    volumes:
      - ./data:/app/data
    depends_on:
      localstack:
        condition: service_healthy

  scheduler:
    image: mcuadros/ofelia:latest
    container_name: local_cloud_scheduler
    volumes:
      - /var/run/docker.sock:/var/run/docker.sock:ro
    command: daemon --docker
    labels:
      ofelia.job-run.fleet-batch-trigger.schedule: "*/1 * * * *"
      ofelia.job-run.fleet-batch-trigger.container: "fleet_batch_worker_instance"

In [ ]:
%%writefile docker-compose.yml
version: '3.8'

services:
  localstack:
    image: localstack/localstack:2.3.2
    container_name: localstack_simulador
    ports:
      - "127.0.0.1:4566:4566"
    environment:
      - SERVICES=s3,sqs,dynamodb  
      - AWS_DEFAULT_REGION=us-east-1
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:4566/_localstack/health"]
      interval: 3s
      timeout: 3s
      retries: 5

  fleet-worker:
    build: .
    image: fleet-multicloud-worker:local
    container_name: fleet_batch_worker_instance
    environment:
      - SIMULACION=False
      - AWS_INPUT_BUCKET=dev-fleet-raw-data
      - AWS_INPUT_KEY=daily_telemetry.csv
      - AZURE_OUTPUT_CONTAINER=dev-fleet-alerts
      - AWS_ACCESS_KEY_ID=mock_key_para_pruebas_locales
      - AWS_SECRET_ACCESS_KEY=mock_secret_para_pruebas_locales
      - AWS_ENDPOINT_URL=http://localstack:4566
    volumes:
      - ./data:/app/data
    depends_on:
      localstack:
        condition: service_healthy

  scheduler:
    image: mcuadros/ofelia:latest
    container_name: local_cloud_scheduler
    volumes:
      - /var/run/docker.sock:/var/run/docker.sock:ro
    command: daemon --docker
    labels:
      ofelia.job-run.fleet-batch-trigger.schedule: "*/1 * * * *"
      ofelia.job-run.fleet-batch-trigger.container: "fleet_batch_worker_instance"

In [ ]:
!docker compose up --build -d

In [ ]:
%%writefile requirements.txt
boto3

In [ ]:
%%writefile app.py
import io
import os
from pathlib import Path
from datetime import datetime
import boto3
from botocore.config import Config
from azure.storage.blob import BlobServiceClient

# =========================================================
# CAPTURA DE VARIABLES DE ENTORNO (EL PUENTE CON DOCKER)
# =========================================================

# os.environ.get busca la variable en Docker. Si no la encuentra, usa el valor por defecto.
# IMPORTANTE: Docker todo lo manda como texto ("True" o "False").
# Por eso lo comparamos con el texto "True" para convertirlo en un Booleano real de Python.
SIMULATION_ENV = os.environ.get("SIMULACION", "True")
SIMULATION_MODE = SIMULATION_ENV.upper() == "TRUE"

# Capturamos el resto de las variables que configuraste en el YAML
AWS_BUCKET = os.environ.get("AWS_INPUT_BUCKET", "fleet-raw-data")
AWS_KEY = os.environ.get("AWS_INPUT_KEY", "raw_telemetry.csv")
AZURE_CONTAINER = os.environ.get("AZURE_OUTPUT_CONTAINER", "fleet-clean-alerts")

print(f"🚀 Live Enterprise Multi-Cloud Pipeline Booting Up...")
print(f"🎛️  Modo de Simulación: {'ACTIVADO (3 filas)' if SIMULATION_MODE else 'DESACTIVADO (1,000 filas con AWS S3)'}")

####################
BASE_DIR = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()
LOCAL_INPUT_ARCHIVE = BASE_DIR / "data" / "archive" / "input"
LOCAL_OUTPUT_ARCHIVE = BASE_DIR / "data" / "archive" / "output"

def s3_telemetry_streamer(s3_client, bucket, key, simulation_mode=False):
    if simulation_mode:
        print("🛠️  [Simulación] Fabricando flujo de datos local en memoria...")
        s3_data = (
            "timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code\n"
            f"{datetime.now()},VIN-999111,2500,95,P0300\n"
            f"{datetime.now()},VIN-222333,1800,88,NONE\n"
            f"{datetime.now()},VIN-444555,3100,102,P0171\n"
        )
    else:
        response = s3_client.get_object(Bucket=bucket, Key=key)
        s3_data = response["Body"].read().decode("utf-8")
    
    LOCAL_INPUT_ARCHIVE.mkdir(parents=True, exist_ok=True)
    timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    local_raw_file = LOCAL_INPUT_ARCHIVE / f"raw_s3_mirror_{timestamp_str}.csv"
    local_raw_file.write_text(s3_data, encoding="utf-8")
    print(f"💾 [Auditoría Local] Réplica cruda de entrada guardada en: {local_raw_file.relative_to(BASE_DIR)}")

    stream_buffer = io.StringIO(s3_data)
    stream_buffer.readline()
    
    for line in stream_buffer:
        if line.strip():
            row = line.strip().split(",")
            yield {
                "timestamp": row[0],
                "vehicle_id": row[1],
                "engine_rpm": int(row[2]),
                "coolant_temp_c": int(row[3]),
                "fault_code": row[4]
            }

def telemetry_processor(data_entry):
    if data_entry["fault_code"] == "NONE":
        return None
    processed_entry = data_entry.copy()
    processed_entry["alert_status"] = "⚠️ CRITICAL DEVIATION DETECTED"
    processed_entry["processed_at"] = str(datetime.now())
    return processed_entry

def azure_bulk_writer(container_client, blob_name, processed_records, simulation_mode=False):
    if not processed_records:
        return
    
    headers = "timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code,alert_status,processed_at\n"
    csv_buffer = headers
    for record in processed_records:
        csv_buffer += (
            f"{record['timestamp']},{record['vehicle_id']},{record['engine_rpm']},"
            f"{record['coolant_temp_c']},{record['fault_code']},{record['alert_status']},"
            f"{record['processed_at']}\n"
        )
    
    if simulation_mode:
        print("☁️  [Simulación] Modo local activo. Se saltó la carga real a Azure.")
    else:
        container_client.upload_blob(name=blob_name, data=csv_buffer, overwrite=True)
        print("☁️  [Cloud] Transmisión exitosa a Azure Blob Storage.")
    
    LOCAL_OUTPUT_ARCHIVE.mkdir(parents=True, exist_ok=True)
    timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    local_clean_file = LOCAL_OUTPUT_ARCHIVE / f"clean_alerts_{timestamp_str}.csv"
    local_clean_file.write_text(csv_buffer, encoding="utf-8")
    print(f"💾 [Auditoría Local] Reporte filtrado guardado en: {local_clean_file.relative_to(BASE_DIR)}")

if __name__ == '__main__':
    AWS_BUCKET = os.getenv("AWS_INPUT_BUCKET", "dev-fleet-raw-data")
    AWS_KEY = os.getenv("AWS_INPUT_KEY", "daily_telemetry.csv")
    AZURE_CONTAINER = os.getenv("AZURE_OUTPUT_CONTAINER", "dev-fleet-alerts")
    
    SIMULACION = os.getenv("SIMULACION", "True").upper() == "TRUE"
    
    s3, azure_container_client = None, None
    
    if not SIMULACION:
        print("⚡ [Modo Enterprise Local] Inicializando entorno simulado de AWS S3 con Moto...")
        
        # 1. ACTIVAR EL MOCK DE MOTO PARA INTERCEPTAR AWS
        from moto import mock_aws
        
        # Iniciamos el contexto simulado de AWS
        mock_context = mock_aws()
        mock_context.start()
        
        # Creamos el cliente S3 virtual apuntando al entorno local en memoria
        s3 = boto3.client("s3", region_name="us-east-1")
        
        # Creamos el bucket virtual y le inyectamos los 1,000 registros ficticios de telemetría
        s3.create_bucket(Bucket=AWS_BUCKET)
        
        # Fabricamos un lote masivo de telemetría simulada (1,000 líneas) para la prueba
        headers = "timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code\n"
        bulk_data = headers
        for i in range(1, 1001):
            code = "P0300" if i % 50 == 0 else "P0171" if i % 75 == 0 else "NONE"
            bulk_data += f"{datetime.now()},VIN-{100000 + i},2100,90,{code}\n"
            
        # Subimos el lote masivo al AWS S3 simulado en memoria
        s3.put_object(Bucket=AWS_BUCKET, Key=AWS_KEY, Body=bulk_data.encode("utf-8"))
        print(f"📦 [S3 Mock] Archivo '{AWS_KEY}' con 1,000 registros cargado con éxito en el Bucket virtual.")
        
        # Simulador local para Azure (Mock)
        class LocalMockAzureContainer:
            def upload_blob(self, name, data, overwrite=True):
                print(f"☁️  [Local Container] Reporte '{name}' interceptado exitosamente en simulación local.")
        
        azure_container_client = LocalMockAzureContainer()
        
    else:
        print("⚠️  Advertencia: Modo de Simulación de 3 filas en memoria Activo.")
    
    print("📥 Connecting to S3 Intake Stream...")
    data_stream = s3_telemetry_streamer(s3, AWS_BUCKET, AWS_KEY, simulation_mode=SIMULACION)
    
    incidents = []
    for raw_row in data_stream:
        clean_row = telemetry_processor(raw_row)
        if clean_row:
            incidents.append(clean_row)
            
    print(f"📤 Preparing handoff for {len(incidents)} processed records...")
    azure_bulk_writer(azure_container_client, "critical_incidents_report.csv", incidents, simulation_mode=SIMULACION)
    
    print("🏁 Batch complete! Multi-cloud architectural loop completed successfully.")

In [ ]:
!docker compose up --build -d

In [ ]:
!docker logs fleet_batch_worker_instance


In [ ]:
# app.py
import os
import boto3  # Este no falla porque está en LocalStack

PROVIDER = os.getenv("CLOUD_PROVIDER", "AWS")

if PROVIDER == "AWS":
    # Inicialización dinámica de AWS para la Fase 5
    s3_client = boto3.client('s3', endpoint_url=os.getenv("AWS_ENDPOINT_URL"))
    print("Conectado con éxito a la infraestructura AWS/LocalStack")

elif PROVIDER == "AZURE":
    # Al meter el import aquí, Python NUNCA lo buscará si estás ejecutando en AWS/LocalStack
    from azure.storage.blob import BlobServiceClient
    
    connect_str = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
    blob_service_client = BlobServiceClient.from_connection_string(connect_str)
    print("Conectado con éxito a la infraestructura Azure")

In [ ]:
!docker compose up --build -d

In [ ]:
!docker logs fleet_batch_worker_instance

In [ ]:

%%writefile docker-compose.yml

version: '3.8'

services:
  # 1. El Simulador de la Infraestructura Cloud (Fase 5)
  localstack_simulador:
    image: localstack/localstack:latest
    container_name: localstack_simulador
    ports:
      - "4566:4566"
    environment:
      - SERVICES=s3,dynamodb
      - AWS_DEFAULT_REGION=us-east-1
    volumes:
      - "localstack_data:/var/lib/localstack"
    healthcheck:
      test: ["CMD-SHELL", "awslocal concepts || exit 0"]
      interval: 5s
      timeout: 5s
      retries: 3

  # 2. El Planificador de Tareas
  local_cloud_scheduler:
    image: alpine:latest
    container_name: local_cloud_scheduler
    command: >
      sh -c "while true; do echo '⏰ [Scheduler] Triggering batch sync...'; sleep 60; done"
    depends_on:
      - localstack_simulador

  # 3. El Worker Principal (Corregido y Nivelado)
  fleet_batch_worker_instance:
    build:
      context: .
      dockerfile: Dockerfile
    container_name: fleet_batch_worker_instance
    restart: always
    environment:
      - CLOUD_PROVIDER=AWS
      # Conexión directa al contenedor de LocalStack usando la red interna de Docker
      - AWS_ENDPOINT_URL=http://localstack_simulador:4566
      - AWS_ACCESS_KEY_ID=mock_key
      - AWS_SECRET_ACCESS_KEY=mock_secret
      - AWS_DEFAULT_REGION=us-east-1
      # Nombre de los componentes de almacenamiento y control definidos
      - S3_BUCKET_NAME=fase5-storage-bucket
      - DYNAMODB_LOCK_TABLE=fase5-execution-locks
    depends_on:
      localstack_simulador:
        condition: service_healthy

volumes:
  localstack_data:

In [ ]:
!docker compose up -d --build

In [ ]:
# 1. Detén y elimina de raíz los contenedores del proyecto anterior que causan el conflicto
!docker rm -f localstack_simulador local_cloud_scheduler fleet_batch_worker_instance

# 2. Levanta el nuevo entorno limpiando cualquier residuo huérfano
!docker compose up -d --build --remove-orphans

In [ ]:
!docker compose up -d --force-recreate

In [ ]:
# 1. Bajamos todo eliminando los volúmenes huérfanos de la prueba anterior
!docker compose down -v

# 2. Levantamos en limpio forzando la recreación
!docker compose up -d --force-recreate

In [ ]:
# 1. Destruye los contenedores y redes anteriores
!docker compose down

# 2. Levanta el entorno ignorando bloqueos de salud
!docker compose up -d --force-recreate

In [ ]:
!docker logs fleet_batch_worker_instance

In [ ]:
# Detiene los contenedores y destruye volúmenes y redes asociadas
!docker compose down -v --remove-orphans

In [ ]:
!ls

In [ ]:
import os
print(os.getcwd())

In [ ]:
%%writefile main.tf


# ==========================================
# CONFIGURACIÓN DEL PROVEEDOR (LOCALSTACK)
# ==========================================
provider "aws" {
  region                      = "us-east-1"
  access_key                  = "mock_access_key"
  secret_key                  = "mock_secret_key"
  skip_credentials_validation = true
  skip_metadata_api_check     = true
  skip_requesting_account_id  = true

  # Apuntamos nativamente al puerto único de LocalStack
  endpoints {
    s3       = "http://localhost:4566"
    dynamodb = "http://localhost:4566"
  }
}

# ==========================================
# ALMACENAMIENTO PERSISTENTE (S3)
# ==========================================
resource "aws_s3_bucket" "storage_bucket" {
  bucket = "fase5-storage-bucket"

  # Nota: En producción usarías aws_s3_bucket_server_side_encryption_configuration
  # Para LocalStack dev, mantenerlo simple acelera el despliegue local.
}

# ==========================================
# CONTROL DE CONCURRENCIA Y BLOQUEOS (DYNAMODB)
# ==========================================
resource "aws_dynamodb_table" "execution_locks" {
  name         = "fase5-execution-locks"
  billing_mode = "PAY_PER_REQUEST" # On-Demand para evitar lidiar con capacidades de lectura/escritura locales
  hash_key     = "LockID"

  attribute {
    name = "LockID"
    type = "S" # String para el identificador del proceso/lote
  }

  tags = {
    Environment = "Local-Dev"
    Phase       = "Phase-5"
  }
}

In [ ]:
!terraform apply  -auto-approve

In [ ]:
!curl http://localhost:4566/_localstack/health

In [ ]:
!docker ps

In [ ]:
!docker-compose up -d

In [ ]:
!docker ps

In [ ]:
!docker logs fleet_batch_worker_instance

In [ ]:
!docker-compose down

In [ ]:
%%writefile main.tf

version: '3.8'

services:
  # 1. El Simulador de la Infraestructura Cloud (Fase 5)
  localstack_simulador:
    image: localstack/localstack:latest
    container_name: localstack_simulador
    ports:
      - "4566:4566"
    environment:
      - AWS_DEFAULT_REGION=us-east-1
    volumes:
      - "localstack_data:/var/lib/localstack"

  # 2. El Planificador de Tareas
  local_cloud_scheduler:
    image: alpine:latest
    container_name: local_cloud_scheduler
    command: >
      sh -c "while true; do echo '⏰ [Scheduler] Triggering batch sync...'; sleep 60; done"
    depends_on:
      - localstack_simulador

  # 3. El Worker Principal (Corregido y Nivelado)
  fleet_batch_worker_instance:
    build:
      context: .
      dockerfile: Dockerfile
    container_name: fleet_batch_worker_instance
    # Cambiado para evitar el bucle infinito si el script termina con éxito (Exit 0)
    restart: on-failure
    environment:
      - CLOUD_PROVIDER=AWS
      - AWS_ENDPOINT_URL=http://localstack_simulador:4566
      - AWS_ACCESS_KEY_ID=mock_key
      - AWS_SECRET_ACCESS_KEY=mock_secret
      - AWS_DEFAULT_REGION=us-east-1
      - S3_BUCKET_NAME=fase5-storage-bucket
      - DYNAMODB_LOCK_TABLE=fase5-execution-locks
    depends_on:
      - localstack_simulador

volumes:
  localstack_data:

In [ ]:
# 1. Apaga el enredo anterior y limpia volúmenes huérfanos
!docker-compose down -v

# 2. Levanta la configuración corregida en segundo plano
!docker-compose up -d

# 3. Verifica que localstack_simulador esté corriendo en el puerto 4566
!docker ps

In [ ]:
!pwd
!ls -la

In [ ]:
%%writefile docker-compose.yml
version: '3.8'

services:
  # 1. El Simulador de la Infraestructura Cloud (Fase 5)
  localstack_simulador:
    image: localstack/localstack:latest
    container_name: localstack_simulador
    ports:
      - "4566:4566"
    environment:
      - AWS_DEFAULT_REGION=us-east-1
    volumes:
      - "localstack_data:/var/lib/localstack"

  # 2. El Planificador de Tareas
  local_cloud_scheduler:
    image: alpine:latest
    container_name: local_cloud_scheduler
    command: >
      sh -c "while true; do echo '⏰ [Scheduler] Triggering batch sync...'; sleep 60; done"
    depends_on:
      - localstack_simulador

  # 3. El Worker Principal (Corregido y Nivelado)
  fleet_batch_worker_instance:
    build:
      context: .
      dockerfile: Dockerfile
    container_name: fleet_batch_worker_instance
    restart: on-failure
    environment:
      - CLOUD_PROVIDER=AWS
      - AWS_ENDPOINT_URL=http://localstack_simulador:4566
      - AWS_ACCESS_KEY_ID=mock_key
      - AWS_SECRET_ACCESS_KEY=mock_secret
      - AWS_DEFAULT_REGION=us-east-1
      - S3_BUCKET_NAME=fase5-storage-bucket
      - DYNAMODB_LOCK_TABLE=fase5-execution-locks
    depends_on:
      - localstack_simulador

volumes:
  localstack_data:

In [ ]:
%%writefile main.tf
# ==========================================
# CONFIGURACIÓN DEL PROVEEDOR (LOCALSTACK)
# ==========================================
provider "aws" {
  region                      = "us-east-1"
  access_key                  = "mock_access_key"
  secret_key                  = "mock_secret_key"
  skip_credentials_validation = true
  skip_metadata_api_check     = true
  skip_requesting_account_id  = true
  
  # Parámetros críticos para evitar atascos con LocalStack
  s3_use_path_style           = true
  skip_region_validation      = true

  endpoints {
    s3       = "http://localhost:4566"
    dynamodb = "http://localhost:4566"
  }
}

# ==========================================
# ALMACENAMIENTO PERSISTENTE (S3)
# ==========================================
resource "aws_s3_bucket" "storage_bucket" {
  bucket = "fase5-storage-bucket"
}

# ==========================================
# CONTROL DE CONCURRENCIA Y BLOQUEOS (DYNAMODB)
# ==========================================
resource "aws_dynamodb_table" "execution_locks" {
  name         = "fase5-execution-locks"
  billing_mode = "PAY_PER_REQUEST"
  hash_key     = "LockID"

  attribute {
    name = "LockID"
    type = "S"
  }

  tags = {
    Environment = "Local-Dev"
    Phase       = "Phase-5"
  }
}

In [ ]:
# 1. Levanta los contenedores con el archivo recién creado
!docker-compose up -d

# 2. Confirma que localstack_simulador esté corriendo estable en el puerto 4566
!docker ps

# 3. Lanza el despliegue de infraestructura limpia
!terraform apply

In [ ]:
!terraform apply -auto-approve

In [ ]:
%%writefile main.tf
# ==========================================
# CONFIGURACIÓN DEL PROVEEDOR (LOCALSTACK CORREGIDO)
# ==========================================
provider "aws" {
  region                      = "us-east-1"
  access_key                  = "mock_access_key"
  secret_key                  = "mock_secret_key"
  skip_credentials_validation = true
  skip_metadata_api_check     = true
  skip_requesting_account_id  = true
  
  s3_use_path_style           = true
  skip_region_validation      = true

  # CAMBIO CLAVE: Usamos la IP IPv4 explícita en lugar de 'localhost'
  endpoints {
    s3       = "http://127.0.0.1:4566"
    dynamodb = "http://127.0.0.1:4566"
  }
}

# ==========================================
# ALMACENAMIENTO PERSISTENTE (S3)
# ==========================================
resource "aws_s3_bucket" "storage_bucket" {
  bucket = "fase5-storage-bucket"
}

# ==========================================
# CONTROL DE CONCURRENCIA Y BLOQUEOS (DYNAMODB)
# ==========================================
resource "aws_dynamodb_table" "execution_locks" {
  name         = "fase5-execution-locks"
  billing_mode = "PAY_PER_REQUEST"
  hash_key     = "LockID"

  attribute {
    name = "LockID"
    type = "S"
  }

  tags = {
    Environment = "Local-Dev"
    Phase       = "Phase-5"
  }
}

In [ ]:
!terraform force-unlock -force

In [ ]:
rm -f .terraform.tfstate.lock.info

In [ ]:
!terraform apply -auto-approve

In [ ]:
!curl -v http://127.0.0.1:4566/_localstack/health

In [ ]:
%%writefile main.tf
# ==========================================
# CONFIGURACIÓN TOTALMENTE BLINDADA PARA LOCALSTACK
# ==========================================
provider "aws" {
  region                      = "us-east-1"
  access_key                  = "mock_access_key"
  secret_key                  = "mock_secret_key"
  
  # Desactivación absoluta de validaciones de AWS Real
  skip_credentials_validation = true
  skip_metadata_api_check     = true
  skip_requesting_account_id  = true
  skip_region_validation      = true
  
  # Forzar estilo de rutas e ignorar firmas SSL locales
  s3_use_path_style           = true

  endpoints {
    s3       = "http://127.0.0.1:4566"
    dynamodb = "http://127.0.0.1:4566"
  }
}

# ==========================================
# ALMACENAMIENTO PERSISTENTE (S3)
# ==========================================
resource "aws_s3_bucket" "storage_bucket" {
  bucket = "fase5-storage-bucket"
}

# ==========================================
# CONTROL DE CONCURRENCIA Y BLOQUEOS (DYNAMODB)
# ==========================================
resource "aws_dynamodb_table" "execution_locks" {
  name         = "fase5-execution-locks"
  billing_mode = "PAY_PER_REQUEST"
  hash_key     = "LockID"

  attribute {
    name = "LockID"
    type = "S"
  }

  tags = {
    Environment = "Local-Dev"
    Phase       = "Phase-5"
  }
}

In [ ]:
rm -f .terraform.tfstate.lock.info

In [ ]:
!docker logs localstack_simulador

In [ ]:
%%writefile docker-compose.yml
version: '3.8'

services:
  # 1. El Simulador de la Infraestructura Cloud (Fase 5 - Versión Comunitaria Libre)
  localstack_simulador:
    image: localstack/localstack-community:latest
    container_name: localstack_simulador
    ports:
      - "4566:4566"
    environment:
      - AWS_DEFAULT_REGION=us-east-1
    volumes:
      - "localstack_data:/var/lib/localstack"

  # 2. El Planificador de Tareas
  local_cloud_scheduler:
    image: alpine:latest
    container_name: local_cloud_scheduler
    command: >
      sh -c "while true; do echo '⏰ [Scheduler] Triggering batch sync...'; sleep 60; done"
    depends_on:
      - localstack_simulador

  # 3. El Worker Principal (Corregido y Nivelado)
  fleet_batch_worker_instance:
    build:
      context: .
      dockerfile: Dockerfile
    container_name: fleet_batch_worker_instance
    restart: on-failure
    environment:
      - CLOUD_PROVIDER=AWS
      - AWS_ENDPOINT_URL=http://localstack_simulador:4566
      - AWS_ACCESS_KEY_ID=mock_key
      - AWS_SECRET_ACCESS_KEY=mock_secret
      - AWS_DEFAULT_REGION=us-east-1
      - S3_BUCKET_NAME=fase5-storage-bucket
      - DYNAMODB_LOCK_TABLE=fase5-execution-locks
    depends_on:
      - localstack_simulador

volumes:
  localstack_data:

In [ ]:
# 1. Destruye los contenedores previos y limpia volúmenes antiguos
!docker-compose down -v

# 2. Levanta la versión comunitaria limpia (descargará la nueva imagen)
!docker-compose up -d

# 3. Dale unos 5 segundos y verifica que el puerto esté abierto de verdad
!curl http://127.0.0.1:4566/_localstack/health

In [ ]:
%%writefile docker-compose.yml
version: '3.8'

services:
  # 1. El Simulador de la Infraestructura Cloud (Fase 5 - Versión Estable Estructurada)
  localstack_simulador:
    image: localstack/localstack:2.3.2
    container_name: localstack_simulador
    ports:
      - "4566:4566"
    environment:
      - AWS_DEFAULT_REGION=us-east-1
      - SERVICES=s3,dynamodb
    volumes:
      - "localstack_data:/var/lib/localstack"

  # 2. El Planificador de Tareas
  local_cloud_scheduler:
    image: alpine:latest
    container_name: local_cloud_scheduler
    command: >
      sh -c "while true; do echo '⏰ [Scheduler] Triggering batch sync...'; sleep 60; done"
    depends_on:
      - localstack_simulador

  # 3. El Worker Principal (Corregido y Nivelado)
  fleet_batch_worker_instance:
    build:
      context: .
      dockerfile: Dockerfile
    container_name: fleet_batch_worker_instance
    restart: on-failure
    environment:
      - CLOUD_PROVIDER=AWS
      - AWS_ENDPOINT_URL=http://localstack_simulador:4566
      - AWS_ACCESS_KEY_ID=mock_key
      - AWS_SECRET_ACCESS_KEY=mock_secret
      - AWS_DEFAULT_REGION=us-east-1
      - S3_BUCKET_NAME=fase5-storage-bucket
      - DYNAMODB_LOCK_TABLE=fase5-execution-locks
    depends_on:
      - localstack_simulador

volumes:
  localstack_data:

In [ ]:
# 1. Bajamos todo limpiando volúmenes antiguos
!docker-compose down -v

# 2. Levantamos descargando explícitamente la versión 2.3.2
!docker-compose up -d

# 3. Espera unos 5-8 segundos a que se inicialicen los servicios internos y prueba la salud:
!curl http://127.0.0.1:4566/_localstack/health

In [ ]:
rm -f .terraform.tfstate.lock.info

In [ ]:
!curl http://127.0.0.1:4566/_localstack/health

In [ ]:
!terraform apply -auto-approve

In [ ]:
%%writefile probar_conexion.py
import boto3
from botocore.exceptions import ClientError
import time

# Configuración del endpoint local
LOCALSTACK_ENDPOINT = "http://127.0.0.1:4566"
TABLE_NAME = "fase5-execution-locks"

print("🔄 Conectando con el servicio DynamoDB en LocalStack...")
dynamodb = boto3.resource(
    'dynamodb',
    endpoint_url=LOCALSTACK_ENDPOINT,
    region_name='us-east-1',
    aws_access_key_id='mock_key',
    aws_secret_access_key='mock_secret'
)

table = dynamodb.Table(TABLE_NAME)

def probar_ciclo_bloqueo():
    lock_id = f"batch_run_{int(time.time())}"
    
    # 1. Intentar adquirir el Lock
    print(f"📥 Intentando adquirir bloqueo con ID: {lock_id}...")
    try:
        table.put_item(
            Item={
                'LockID': lock_id,
                'Status': 'PROCESSING',
                'Timestamp': str(time.time())
            },
            # Esta condición asegura que el bloqueo falle si el ID ya existe
            ConditionExpression='attribute_not_exists(LockID)'
        )
        print("✅ ¡Bloqueo ADQUIRIDO con éxito en DynamoDB!")
        
    except ClientError as e:
        if e.response['Error']['Code'] == 'ConditionalCheckFailedException':
            print("❌ Error: El bloqueo ya existe.")
        else:
            print(f"❌ Error inesperado al adquirir: {e}")
        return

    # Espera simulada de procesamiento corta
    time.sleep(2)

    # 2. Intentar liberar el Lock
    print(f"📤 Intentando liberar el bloqueo: {lock_id}...")
    try:
        table.delete_item(
            Key={'LockID': lock_id}
        )
        print("✅ ¡Bloqueo LIBERADO con éxito! La tabla quedó limpia.")
    except ClientError as e:
        print(f"❌ Error al liberar el bloqueo: {e}")

if __name__ == "__main__":
    try:
        # Verificar primero si la tabla existe
        table.load()
        print(f"📊 Tabla '{TABLE_NAME}' detectada correctamente.")
        probar_ciclo_bloqueo()
    except ClientError as e:
        print(f"❌ No se pudo acceder a la tabla. ¿Corriste Terraform? Detalles: {e}")

In [ ]:
!python probar_conexion.py

In [ ]:
import boto3
from botocore.exceptions import ClientError
import time

# Configuración del endpoint local
LOCALSTACK_ENDPOINT = "http://127.0.0.1:4566"
TABLE_NAME = "fase5-execution-locks"

print("🔄 Conectando con el servicio DynamoDB en LocalStack...")
dynamodb = boto3.resource(
    'dynamodb',
    endpoint_url=LOCALSTACK_ENDPOINT,
    region_name='us-east-1',
    aws_access_key_id='mock_key',
    aws_secret_access_key='mock_secret'
)

table = dynamodb.Table(TABLE_NAME)

def probar_ciclo_bloqueo():
    lock_id = f"batch_run_{int(time.time())}"
    
    # 1. Intentar adquirir el Lock
    print(f"📥 Intentando adquirir bloqueo con ID: {lock_id}...")
    try:
        table.put_item(
            Item={
                'LockID': lock_id,
                'Status': 'PROCESSING',
                'Timestamp': str(time.time())
            },
            # Esta condición asegura que el bloqueo falle si el ID ya existe
            ConditionExpression='attribute_not_exists(LockID)'
        )
        print("✅ ¡Bloqueo ADQUIRIDO con éxito en DynamoDB!")
        
    except ClientError as e:
        if e.response['Error']['Code'] == 'ConditionalCheckFailedException':
            print("❌ Error: El bloqueo ya existe.")
        else:
            print(f"❌ Error inesperado al adquirir: {e}")
        return

    # Espera simulada de procesamiento corta
    time.sleep(2)

    # 2. Intentar liberar el Lock
    print(f"📤 Intentando liberar el bloqueo: {lock_id}...")
    try:
        table.delete_item(
            Key={'LockID': lock_id}
        )
        print("✅ ¡Bloqueo LIBERADO con éxito! La tabla quedó limpia.")
    except ClientError as e:
        print(f"❌ Error al liberar el bloqueo: {e}")

if __name__ == "__main__":
    try:
        # Verificar primero si la tabla existe
        table.load()
        print(f"📊 Tabla '{TABLE_NAME}' detectada correctamente.")
        probar_ciclo_bloqueo()
    except ClientError as e:
        print(f"❌ No se pudo acceder a la tabla. ¿Corriste Terraform? Detalles: {e}")


In [ ]:
!docker-compose up -d

In [ ]:
!curl http://127.0.0.1:4566/health

In [ ]:
!python probar_conexion.py

In [ ]:
!terraform init

In [ ]:
!terraform plan

In [ ]:
!terraform apply -auto-approve

In [ ]:
!python probar_conexion.py

In [ ]:
# Reinicia el worker para que tome el flujo limpio contra el LocalStack que acabas de poblar
!docker compose up -d --build fleet_batch_worker_instance

In [ ]:
!docker compose logs -f fleet_batch_worker_instance

In [ ]:
# 1. Levanta y reconstruye el worker con la nueva variable de red
!docker compose up -d --build fleet_batch_worker_instance

# 2. Sigue los logs en tiempo real para ver el procesamiento de las filas
!docker compose logs -f fleet_batch_worker_instance

In [ ]:
!docker compose up -d --force-recreate fleet_batch_worker_instance
!docker compose logs -f fleet_batch_worker_instance

In [ ]:
# 1. Detén por completo todos los servicios actuales
!docker compose down

# 2. Levanta todo de nuevo recreando el entorno limpio
!docker compose up -d --build

# 3. Mira los logs del worker para ver correr tus datos
!docker compose logs -f fleet_batch_worker_instance

In [ ]:
# 1. Detén y destruye contenedores anteriores
!docker compose down

# 2. Levanta obligando a Docker a reconstruir el código de Python sin usar caché
!docker compose build --no-cache fleet_batch_worker_instance

# 3. Enciende el entorno
!docker compose up -d

# 4. Sigue los logs
!docker compose logs -f fleet_batch_worker_instance

In [ ]:
# Forzamos la reconstrucción con el código corregido
!docker compose up -d --build fleet_batch_worker_instance

# Seguimos el flujo
!docker compose logs -f fleet_batch_worker_instance

In [ ]:
# 1. Destruye los contenedores y las redes aisladas viejas
!docker compose down

# 2. Levanta todo el ecosistema usando la nueva red unificada
!docker compose up -d

# 3. Sigue los logs de tu worker
!docker compose logs -f fleet_batch_worker_instance

In [ ]:
localstack_simulador:
    image: localstack/localstack:2.3.2
    container_name: localstack_simulador
    ports:
      - "4566:4566"
    environment:
      AWS_DEFAULT_REGION: us-east-1
      SERVICES: s3,dynamodb
      EDGE_BIND_HOST: 0.0.0.0
    volumes:
      - "localstack_data:/var/lib/localstack"
    networks:
      - fase5_network

In [ ]:
!docker compose down -v
!docker compose up -d --build
!docker compose logs -f fleet_batch_worker_instance

In [ ]:
! Finalmente applicacion funcionando!!!!!!

In [ ]:
!docker compose up -d --build

In [ ]:
!docker compose logs --tail=50 fleet_batch_worker_instance

In [ ]:
!docker compose up -d --build fleet_batch_worker_instance
!docker compose logs -f fleet_batch_worker_instance

In [ ]:
!docker compose up -d --build fleet_batch_worker_instance
!docker compose logs -f fleet_batch_worker_instance

In [9]:
#!/bin/bash
awslocal dynamodb create-table \
    --table-name TuNombreDeTabla \
    --attribute-definitions AttributeName=LockID,AttributeType=S \
    --key-schema AttributeName=LockID,KeyType=HASH \
    --provisioned-throughput ReadCapacityUnits=5,WriteCapacityUnits=5

SyntaxError: invalid syntax (4062012753.py, line 2)

In [ ]:
!docker compose up --build

In [22]:
!docker compose logs -f # corre superbien


^C
canceled


In [ ]:
!docker compose logs -f fleet_batch_worker_instance

In [17]:
!docker stats

CONTAINER ID   NAME                   CPU %     MEM USAGE / LIMIT     MEM %     NET I/O          BLOCK I/O         PIDS
2ddcfe6bdf5f   localstack_simulador   3.43%     566.1MiB / 3.842GiB   14.39%    1.1MB / 6.01MB   63.1MB / 8.76MB   63
CONTAINER ID   NAME                   CPU %     MEM USAGE / LIMIT     MEM %     NET I/O          BLOCK I/O         PIDS
2ddcfe6bdf5f   localstack_simulador   0.75%     566.1MiB / 3.842GiB   14.39%    1.1MB / 6.01MB   63.1MB / 8.76MB   63
CONTAINER ID   NAME                   CPU %     MEM USAGE / LIMIT     MEM %     NET I/O          BLOCK I/O         PIDS
2ddcfe6bdf5f   localstack_simulador   0.75%     566.1MiB / 3.842GiB   14.39%    1.1MB / 6.01MB   63.1MB / 8.76MB   63
CONTAINER ID   NAME                   CPU %     MEM USAGE / LIMIT     MEM %     NET I/O          BLOCK I/O         PIDS
2ddcfe6bdf5f   localstack_simulador   0.75%     566.1MiB / 3.842GiB   14.39%    1.1MB / 6.01MB   63.1MB / 8.76MB   63
CONTAINER ID   NAME                   CPU %     

In [16]:
!aws --endpoint-url=http://localhost:4566 s3 ls s3://fase5-storage-bucket/processed_outputs/
# Descargar un reporte procesado para auditarlo visualmente:

/bin/bash: aws: command not found


In [18]:
!aws --endpoint-url=http://localhost:4566 s3 cp s3://fase5-storage-bucket/processed_outputs/NOMBRE_DEL_ARCHIVO.csv .


/bin/bash: aws: command not found


In [19]:
!aws --endpoint-url=http://localhost:4566 dynamodb scan --table-name fase5-execution-locks


/bin/bash: aws: command not found


In [20]:
!terraform show

# aws_dynamodb_table.execution_locks:
resource "aws_dynamodb_table" "execution_locks" {
    arn                         = "arn:aws:dynamodb:us-east-1:000000000000:table/fase5-execution-locks"
    billing_mode                = "PAY_PER_REQUEST"
    deletion_protection_enabled = false
    hash_key                    = "LockID"
    id                          = "fase5-execution-locks"
    name                        = "fase5-execution-locks"
    read_capacity               = 0
    stream_enabled              = false
    table_class                 = "STANDARD"
    tags                        = {
        "Environment" = "Local-Dev"
        "Phase"       = "Phase-5"
    }
    tags_all                    = {
        "Environment" = "Local-Dev"
        "Phase"       = "Phase-5"
    }
    write_capacity              = 0

    attribute {
        name = "LockID"
        type = "S"
    }

    point_in_time_recovery {
        enabled = false
    }

    ttl {
        enabled = false
    }
}

# aws_

In [21]:
!terraform plan

aws_dynamodb_table.execution_locks: Refreshing state... [id=fase5-execution-locks]
aws_s3_bucket.storage_bucket: Refreshing state... [id=fase5-storage-bucket]

Terraform used the selected providers to generate the following execution plan.
Resource actions are indicated with the following symbols:
  ~ update in-place

Terraform will perform the following actions:

  # aws_dynamodb_table.execution_locks will be updated in-place
  ~ resource "aws_dynamodb_table" "execution_locks" {
      ~ billing_mode                = "PROVISIONED" -> "PAY_PER_REQUEST"
        id                          = "fase5-execution-locks"
        name                        = "fase5-execution-locks"
      ~ tags                        = {
          + "Environment" = "Local-Dev"
          + "Phase"       = "Phase-5"
        }
      ~ tags_all                    = {
          + "Environment" = "Local-Dev"
          + "Phase"       = "Phase-5"
        }
        # (7 unchanged attributes hidden)

        # (3 unchan